# Agentic pipeline experiment

Offline notebook for Wave B stages. Run from `services/agentic-pipeline` with deps installed.

```bash
pip install -r requirements.txt jupyter
jupyter notebook notebooks/01_pipeline_experiment.ipynb
```

In [ ]:
import sys
from pathlib import Path
root = Path.cwd()
if (root / "app").exists():
    sys.path.insert(0, str(root))
elif (root.parent / "app").exists():
    sys.path.insert(0, str(root.parent))

from app.models import PageDoc, PipelineState
from app.stages.analyzer import run_analyzer
from app.stages.preprocessor import run_preprocessor
from app.stages.optimizer import run_optimizer
from app.stages.validator import run_validator

sample = """# Example Domain

This domain is for use in illustrative examples in documents.
You may use this domain in literature without prior coordination or asking for permission.
"""

state = PipelineState(url="https://example.com", question="What is example domain for?")
state.pages = [PageDoc(url="https://example.com", markdown=sample)]
state = run_analyzer(state)
state = run_preprocessor(state)
state = run_optimizer(state)
print("chunks", len(state.chunks), "ranked", len(state.ranked_chunks))
print("bm25_top", state.scores.get("bm25_top"))

state.draft = (
    "Example Domain is for use in illustrative examples in documents "
    "without prior coordination or asking for permission."
)
state = run_validator(state)
print("validation", state.validation)

In [ ]:
# Optional: full async pipeline (uses stub LLM when no API keys)
import asyncio
from app.pipeline import run_pipeline

async def main():
    # Monkeypatch-free path hits network for extract; for offline, paste pages via stages above.
    print("Use run_pipeline only when network/extractors are available.")

asyncio.run(main())